# Macroeconomic revision risk

This notebook asks whether classifying growth with today's revised history materially
changes the cross-asset differences obtained from information available in real time.
Revisions are treated as retrospective diagnostics, never as contemporaneously
observable features.

The real-GDP threshold, payroll sensitivity, outcomes, and baselines were fixed before
execution. No observed provider values, tables, plots, or conclusions are saved in the
committed notebook.

**Primary protocol.** The decision is the last common ETF trading session of a complete
month, using macro information available one calendar day earlier. Faster GDP growth is
the treatment, slower growth the reference, and one month the primary horizon. For each
core ETF, the primary revision estimand is the **signed** latest-revised
faster-minus-slower contrast minus the signed point-in-time contrast on dates classified
in both views. No direction is predeclared. Bonferroni intervals cover the five-asset
family. A total revision-plus-availability estimate, longer horizons, GDP thresholds,
payrolls, and temporal splits are exploratory. “Cleaner” or absolute separation is not
claimed.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

from studies._support import (
    CORE_SYMBOLS,
    STUDY_START,
    acquire_latest_series,
    acquire_monthly_prices,
    acquire_vintage_histories,
    assert_component_periods_match,
    assert_revision_change_identities,
    build_point_in_time_levels,
    classification_change_summary,
    classification_transition_table,
    compare_statistics,
    component_availability_summary,
    configure_plots,
    feature_provenance_summary,
    forward_labels,
    latest_revised_year_over_year,
    momentum_baseline,
    open_live_session,
    plot_coverage,
    plot_feature_comparison,
    plot_normalized_prices,
    plot_regime_contrasts,
    plot_regime_distributions,
    plot_regime_means,
    plot_regime_timeline,
    plot_revision_contrast_change,
    plot_revision_gap,
    plot_sample_sizes,
    plot_sensitivity_heatmap,
    point_in_time_year_over_year,
    regime_contrast_statistics,
    regime_statistics,
    regime_style,
    revision_contrast_change,
    simultaneous_interval_family,
    study_run_manifest,
    temporal_revision_change_stability,
    unconditional_statistics,
    validate_study_outputs,
)

configure_plots()
pd.set_option("display.max_columns", 20)
session = open_live_session()

## Hypothesis, series, and fixed market universe

`GDPC1` is quarterly real gross domestic product. It is the primary series because
benchmark revisions can alter the historical growth path. `PAYEMS` is monthly payroll
employment and supplies a higher-frequency sensitivity. Both are revised and both are
selected through ALFRED availability intervals.

The core ETF universe spans equities, intermediate Treasuries, gold, commodities, and
the dollar. The hypothesis is that latest-revised classification can change the signed
faster-minus-slower return contrast. A second estimate restricts both views to the same
classified dates, while availability transitions are reported separately. A contrary,
near-zero, or unavailable change remains informative about this sample.

Real GDP is a chain-type quantity measure, so benchmark updates can revise long spans
and reference years. Payroll employment measures a different monthly concept; it is a
sensitivity, not a validation target. The live normalized objects round-trip through a
temporary DuckDB database before any transformation.

In [ ]:
price_history, market_provenance = acquire_monthly_prices(session, CORE_SYMBOLS)
prices = price_history.loc[STUDY_START:]
series_ids = ("GDPC1", "PAYEMS")
histories = acquire_vintage_histories(session, series_ids, prices.index)
latest = acquire_latest_series(session, series_ids)
staleness = {
    "GDPC1": pd.Timedelta(days=150),
    "PAYEMS": pd.Timedelta(days=62),
}
point_in_time = build_point_in_time_levels(
    histories,
    prices.index,
    staleness,
    observation_date_columns={"GDPC1": "period_end"},
)
feature_provenance = feature_provenance_summary(point_in_time)
manifest = study_run_manifest(
    prices,
    series_ids=series_ids,
    thresholds="GDP growth=2%; payroll sensitivity=0%",
    staleness=staleness,
)
display(manifest, market_provenance, feature_provenance)

## Unequal frequency and stale releases

Monthly decisions do not turn quarterly GDP into monthly information. Between GDP
releases, the newest admissible observation is carried only within a 150-day
staleness ceiling, and provenance records the matched observation age. Payrolls use a
shorter ceiling. The coverage plot exposes these policies rather than hiding them
behind forward filling.

GDP observation age is measured from the derived quarter end, not the quarter start.
This allows the 150-day ceiling to span ordinary release spacing without turning the
sample into release months only. The cutoff must still lie inside the selected version's
availability interval. Observation period, availability, and this execution's retrieval
time remain separate provenance fields.

ETF inception and missing provider sessions affect the market panel independently of
macro release coverage. Joint outcome counts are examined later.

In [ ]:
figure, _ = plot_coverage(market_provenance, feature_provenance)
plt.show()
plt.close(figure)

## Real-time and latest-revised growth paths

Year-over-year growth is calculated within one complete vintage known at each decision.
For newest source quarter \(t\), GDP growth is
\(100(G_{d,t}/G_{d,t-4}-1)\); the exact year-ago quarter comes from the same snapshot.
Payroll growth analogously uses exact source months \(t\) and \(t-12\). Repeated monthly
decisions can refer to the same quarterly observation, but they never substitute twelve
decision rows for four source quarters.

Today's revised counterpart keeps the exact component periods and substitutes current
values. Component provenance records actual period, availability, source identity,
definition fields, and retrieval time. This avoids arithmetic across two benchmark
vintages while making component availability changes auditable.

Plotting both GDP and payroll growth distinguishes the primary quarterly feature from
the monthly sensitivity before any return grouping occurs.

In [ ]:
point_growth_result = point_in_time_year_over_year(histories, point_in_time)
latest_growth_result = latest_revised_year_over_year(point_in_time, latest)
point_growth = point_growth_result.frame
latest_growth = latest_growth_result.frame
point_features = point_growth.rename(
    columns={"GDPC1": "Real GDP growth", "PAYEMS": "Payroll growth"}
)
latest_features = latest_growth.set_axis(point_features.columns, axis="columns")
figure, _ = plot_feature_comparison(
    point_features,
    latest_features,
    tuple(point_features.columns),
)
plt.show()
plt.close(figure)

## Primary classification and leakage boundary

Faster real-GDP growth is above two percent year over year; slower growth is at or
below two percent. The threshold is fixed and intentionally broader than a recession
rule. It yields an interpretable growth comparison without claiming that a particular
rate is structurally optimal.

The point-in-time classification is the only primary regime. Later revisions are
evaluation information, analogous to a label: they can describe how a past conclusion
changed but cannot enter the decision-date feature.

In [ ]:
def growth_regime(growth: pd.Series, *, boundary: float = 2.0) -> pd.Series:
    regime = pd.Series(pd.NA, index=growth.index, dtype="string")
    regime.loc[growth.notna() & growth.gt(boundary)] = "faster growth"
    regime.loc[growth.notna() & growth.le(boundary)] = "slower growth"
    return regime

point_regimes = growth_regime(point_growth["GDPC1"])
latest_regimes = growth_regime(latest_growth["GDPC1"])
display(point_regimes.value_counts(dropna=False).rename("decision count"))
figure, _ = plot_regime_timeline(
    point_growth["GDPC1"],
    point_regimes,
    title="Point-in-time real-GDP growth classification",
    ylabel="Year-over-year percent change",
    boundaries=(2.0,),
)
plt.show()
plt.close(figure)

figure, axis = plt.subplots(figsize=(7, 7))
valid = point_growth["GDPC1"].notna() & latest_growth["GDPC1"].notna()
for regime in ("faster growth", "slower growth"):
    color, marker = regime_style(regime)
    selected = valid & point_regimes.eq(regime).fillna(False)
    axis.scatter(
        point_growth.loc[selected, "GDPC1"],
        latest_growth.loc[selected, "GDPC1"],
        label=regime,
        marker=marker,
        color=color,
        alpha=0.75,
    )
combined = pd.concat(
    [point_growth.loc[valid, "GDPC1"], latest_growth.loc[valid, "GDPC1"]]
)
limits = (combined.min(), combined.max())
axis.plot(limits, limits, color="#333333", linestyle="--", linewidth=1)
axis.set_xlim(limits)
axis.set_ylim(limits)
axis.set_aspect("equal", adjustable="box")
axis.set(
    title="Real-time and latest-revised GDP growth",
    xlabel="Point-in-time year-over-year growth (percent)",
    ylabel="Latest-revised year-over-year growth (percent)",
)
axis.legend()
figure.tight_layout()
plt.show()
plt.close(figure)

## Separated outcomes and baselines

One-, three-, and twelve-month asset returns are computed only from the Alpha Vantage
price panel and stored in typed label objects. The final incomplete horizons remain
missing, and each label carries its ending date. Macro features and asset labels have
disjoint columns and construction paths.

Unconditional outcomes and a trailing twelve-month price-momentum split provide two
conventional benchmarks. Normalized closes show market history but do not represent a
regime-switching portfolio.

The return label is \(R_{d,h}=P_{d+h}/P_d-1\) and stores its actual ending close. Live
checks require an exact \(h\)-calendar-month period offset and leave terminal outcomes
missing. Both baselines use point-in-time GDP-regime-eligible dates. One year of earlier
prices supplies the twelve-month momentum feature at the first analysis date.

In [ ]:
labels = forward_labels(prices)
eligible = point_regimes.notna()
unconditional = unconditional_statistics(labels, eligible=eligible)
momentum = momentum_baseline(price_history, labels, eligible=eligible)
display(unconditional, momentum)
figure, _ = plot_normalized_prices(prices)
plt.show()
plt.close(figure)

## Conditional summaries and uncertainty

The primary table contains the complete asset-by-regime-by-horizon family. Counts,
coverage, means, horizon-return dispersion, positive shares,
heteroskedasticity-and-autocorrelation-consistent (HAC) standard errors, confidence
bounds,
and one-month episode-aware drawdowns remain visible. Longer-horizon overlap is
reflected in the minimum bandwidth, while an automatic rule permits additional serial
covariance. One-month annualized volatility is reported in a separate column.

Descriptive group intervals are pointwise. The five primary signed revision changes use
Bonferroni simultaneous intervals, paired through aligned influence series so the
covariance between the two classifications is retained. The display rule requires
twelve outcomes and two outcome-eligible episodes per side; it is not a claim that two
episodes make normal inference adequate. First/second-half and leave-one-episode-out
results remain necessary context.

Comparing the latest-revised table with the primary table is a specification-bias
audit. It is not an invitation to use the revised table as a historical strategy.

In [ ]:
point_statistics = regime_statistics(labels, point_regimes)
latest_statistics = regime_statistics(labels, latest_regimes)
point_contrasts = regime_contrast_statistics(
    labels,
    point_regimes,
    treated="faster growth",
    reference="slower growth",
    assets=CORE_SYMBOLS,
)
total_revision_change = revision_contrast_change(
    labels,
    point_regimes,
    latest_regimes,
    treated="faster growth",
    reference="slower growth",
)
common_sample_revision_change = revision_contrast_change(
    labels,
    point_regimes,
    latest_regimes,
    treated="faster growth",
    reference="slower growth",
    common_classified_only=True,
)
display(
    point_statistics,
    point_contrasts,
    total_revision_change,
    common_sample_revision_change,
)
figure, _ = plot_regime_means(
    point_statistics,
    title="Real-time GDP growth and one-month outcomes",
)
plt.show()
plt.close(figure)

figure, _ = plot_regime_contrasts(
    point_contrasts,
    title="Real-time faster-minus-slower growth contrasts",
)
plt.show()
plt.close(figure)

figure, _ = plot_revision_contrast_change(
    common_sample_revision_change,
    title="Signed contrast change on dates classified in both views",
)
plt.show()
plt.close(figure)

## Distributions and effective sample size

Growth regimes cluster through expansions, slowdowns, and contractions. A large count
of monthly decisions can therefore reflect fewer independent macro episodes. Box
plots show the distribution behind each mean, while the count chart exposes state
imbalance and incomplete forward horizons.

Provider observations are retained as received after validation. No outlier is removed
because it weakens or strengthens the predeclared hypothesis.

Box plots show quartiles and retain every tail point. Count bars distinguish monthly
outcomes from contiguous, outcome-eligible GDP episodes. Repeating one quarterly state
over several monthly decisions can still make effective information diversity smaller
than either number suggests, which is why leave-one-episode-out estimates are shown.

In [ ]:
figure, _ = plot_regime_distributions(
    labels[1],
    point_regimes,
    assets=("SPY", "IEF", "GLD", "DBC"),
    title="One-month outcomes by real-time GDP growth state",
)
plt.show()
plt.close(figure)

figure, _ = plot_sample_sizes(point_statistics)
plt.show()
plt.close(figure)

## Threshold and payroll sensitivity

The GDP grid uses boundaries of zero, one, two, and three percent. Every cell reports
the faster-minus-slower one-month mean for one core asset. The separate payroll
sensitivity classifies positive versus nonpositive year-over-year employment growth.
Both analyses are displayed in full; neither is selected by return magnitude.

A pattern that depends on one GDP boundary or disappears with payroll growth is a
fragility to report, not a prompt for another search.

The GDP threshold table retains counts, episode counts, HAC errors, and intervals, and
one exploratory Bonferroni family covers all twenty threshold-asset cells. Masked cells
fail the minimum-data display rule; they do not mean zero. Payroll repeats both the
total revision-plus-availability change and the common-classified-date signed change,
so the monthly sensitivity tests the revision-risk question rather than merely showing
another regime table.

In [ ]:
sensitivity_rows = []
for boundary in (0.0, 1.0, 2.0, 3.0):
    regimes = growth_regime(point_growth["GDPC1"], boundary=boundary)
    table = regime_contrast_statistics(
        {1: labels[1]},
        regimes,
        treated="faster growth",
        reference="slower growth",
        assets=CORE_SYMBOLS,
    )
    table.insert(0, "boundary", boundary)
    sensitivity_rows.append(table)
sensitivity_table = simultaneous_interval_family(
    pd.concat(sensitivity_rows, ignore_index=True),
    family_id="GDP threshold family",
)
sensitivity = sensitivity_table.pivot(
    index="boundary", columns="asset", values="mean_difference"
).where(
    sensitivity_table.pivot(
        index="boundary", columns="asset", values="meets_display_threshold"
    )
)
payroll_regimes = growth_regime(point_growth["PAYEMS"], boundary=0.0)
latest_payroll_regimes = growth_regime(latest_growth["PAYEMS"], boundary=0.0)
payroll_statistics = regime_statistics(labels, payroll_regimes)
payroll_revision_change = revision_contrast_change(
    labels,
    payroll_regimes,
    latest_payroll_regimes,
    treated="faster growth",
    reference="slower growth",
)
common_payroll_revision_change = revision_contrast_change(
    labels,
    payroll_regimes,
    latest_payroll_regimes,
    treated="faster growth",
    reference="slower growth",
    common_classified_only=True,
)
display(
    sensitivity_table,
    sensitivity,
    payroll_statistics,
    payroll_revision_change,
    common_payroll_revision_change,
)
figure, _ = plot_sensitivity_heatmap(
    sensitivity,
    title="Faster-minus-slower contrasts across GDP boundaries",
    color_label="Difference in one-month mean return",
)
plt.show()
plt.close(figure)

## Revision magnitude and classification instability

The latest-revised diagnostic measures changes in selected growth values, regime
membership, sample counts, and conditional means. A classification cross-tab makes
boundary crossings explicit. The revision-gap figure shows the retrospective GDP
growth difference without feeding it back into the feature panel.

The hypothesis concerns whether revised history changes descriptive conclusions. It
does not claim that future revision direction can be traded.

“Signed contrast change” is defined as
\((\bar R_F-\bar R_S)_{latest}-(\bar R_F-\bar R_S)_{real\ time}\). It is not the change
in absolute magnitude. Aligned influence contributions retain covariance between the
two classifications. The primary common-sample table requires a classification in both
views. A separate total table allows component availability to change, while the
component summary, reclassification denominator and rate, and unclassified transition
cells identify that channel explicitly. The plotted simultaneous intervals use the
common-sample version.

In [ ]:
revision_comparison = compare_statistics(point_statistics, latest_statistics)
classification_changes = classification_transition_table(
    point_regimes,
    latest_regimes,
)
classification_summary = classification_change_summary(
    point_regimes,
    latest_regimes,
)
component_availability = component_availability_summary(
    point_growth_result,
    latest_growth_result,
)
revision_diagnostics = pd.DataFrame(
    {
        "point-in-time growth": point_growth["GDPC1"],
        "latest-revised growth": latest_growth["GDPC1"],
        "classification changed": point_regimes.ne(latest_regimes),
    }
)
stability = temporal_revision_change_stability(
    labels,
    point_regimes,
    latest_regimes,
    treated="faster growth",
    reference="slower growth",
    assets=CORE_SYMBOLS,
)
display(
    revision_comparison,
    classification_changes,
    classification_summary,
    component_availability,
    total_revision_change,
    common_sample_revision_change,
    revision_diagnostics,
    stability,
)
figure, _ = plot_revision_gap(
    point_growth["GDPC1"],
    latest_growth["GDPC1"],
    title="Real-GDP growth revision substitution gap",
)
plt.show()
plt.close(figure)

## Limitations and adversarial assertions

Quarterly GDP is released with delay and revised through annual and benchmark cycles.
A monthly decision grid repeats quarterly information. Payroll employment captures a
different concept and is not a validation target for GDP. The ETF universe has
inception, fixed-selection, and survivorship bias. Returns exclude costs, taxes, and
executable release timing. Overlapping labels, clustered growth regimes, approximate
intervals, and the full asset-horizon-threshold family create multiple-testing risk.

Live assertions verify temporal availability, provenance completeness, label
separation, index alignment, multiple regimes, and finite outputs.

Current FRED definition fields are applied to historical vintage rows and cannot prove
that old benchmark metadata were unchanged. Within-vintage ratios avoid mixing two
definitions, but revision interpretation still includes methodological updates. BEA's
[GDP release information](https://www.bea.gov/news/gdp-release-additional-information)
explains advance, second, third, annual, and comprehensive updates. The
[BEA methodology index](https://www.bea.gov/resources/methodologies) provides deeper
national-account concepts. ALFRED supplies historical information sets; none of these
sources makes revisions observable at the earlier market decision.

In [ ]:
audit = validate_study_outputs(
    prices,
    point_in_time,
    labels,
    point_regimes,
    point_statistics,
    expected_regimes=frozenset({"faster growth", "slower growth"}),
    transformed=point_growth_result,
)
assert_component_periods_match(point_growth_result, latest_growth_result)
assert_revision_change_identities(
    labels,
    point_regimes,
    treated="faster growth",
    reference="slower growth",
)
assert set(point_in_time.frame.columns).isdisjoint(labels[1].frame.columns)
matched = point_in_time.provenance["available_from"].notna()
assert point_in_time.provenance.loc[matched, "available_from"].le(
    point_in_time.provenance.loc[matched, "decision_date"] - pd.Timedelta(days=1)
).all()
display(audit)
session.close()

## Interpretation after execution

Start with matched observation ages, feature coverage, and regime counts. Compare
point-in-time distributions and uncertainty with both baselines, then inspect the full
GDP grid, payroll sensitivity, classification cross-tab, and revised-history changes.
Preserve negative and unstable evidence. This notebook studies revision-sensitive
association; it does not prove causality, forecast skill, or strategy profitability.